# Evaluation Code

In this notebook, we will demonstrate how our two metrics, ROUGE and BERTScore, function. Evaluating the quality of generated summaries is not as straightforward as other machine learning tasks. Both metrics have inherent limitations that we will address below.

In [13]:
# Install necessary libraries

!pip install -q rouge_score bert_score

In [14]:
# Import necessary libraries

import pandas as pd
import numpy as np
from rouge_score import rouge_scorer
from bert_score import score

import warnings
warnings.filterwarnings('ignore')

## ROUGE

The main benefit of ROUGE is that it provides a simple and interpretable way to evaluate summarization quality based on word/phrase overlap with reference texts. However, the metric struggles to capture semantic similarity when different words convey the same meaning, making it less reliable for evaluating paraphrased or more abstractive summaries.

In [15]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

# Dummy gold summary
gold = "This phone has a great camera and its battery lasts long."

# Candidate 1: high lexical overlap
candidate_1 = "The phone's camera is great and lasts a long time."
scores_1 = scorer.score(gold, candidate_1)

# Candidate 2: same meaning, different words
candidate_2 = "My device takes good photos, and I don't have to charge it very often."
scores_2 = scorer.score(gold, candidate_2)

def round_scores(score_dict):
    return {k: {
        "precision": round(v.precision, 3),
        "recall": round(v.recall, 3),
        "fmeasure": round(v.fmeasure, 3)
    } for k, v in score_dict.items()}

scores_1 = round_scores(scores_1)
scores_2 = round_scores(scores_2)

# Print scores
print("Candidate 1 (high word overlap):")
print(f"ROUGE-1: {scores_1['rouge1']}")
print(f"ROUGE-L: {scores_1['rougeL']}\n")

print("Candidate 2 (semantic similarity, low word overlap):")
print(f"ROUGE-1: {scores_2['rouge1']}")
print(f"ROUGE-L: {scores_2['rougeL']}")

Candidate 1 (high word overlap):
ROUGE-1: {'precision': 0.636, 'recall': 0.636, 'fmeasure': 0.636}
ROUGE-L: {'precision': 0.455, 'recall': 0.455, 'fmeasure': 0.455}

Candidate 2 (semantic similarity, low word overlap):
ROUGE-1: {'precision': 0.067, 'recall': 0.091, 'fmeasure': 0.077}
ROUGE-L: {'precision': 0.067, 'recall': 0.091, 'fmeasure': 0.077}


While both candidate summaries describe the same qualities of the product, candidate 1 has much higher ROUGE scores due to its similar wording to the gold summary. This poses a problem in review summarization as the precise wording does not matter. BERTScore helps address this issue.

## BERTScore

In order to score generated summaries beyond exact word matches, BERTScore uses contextual embeddings to capture semantic similarity. This is especially important for abstractive summaries, where the language used may differ even if the details are accurate. Since this metric is based on a large pre-trained language model, it's much slower to compute and is less interpretable than ROUGE.

In [16]:
# Dummy gold summary
gold = ["This phone has a great camera and its battery lasts long."]

# Candidate 1: high lexical overlap
candidate_1 = ["The phone's camera is great and lasts a long time."]

# Candidate 2: same meaning, different words
candidate_2 = ["My device takes good photos, and I don't have to charge it very often."]

# Compute BERTScore
P1, R1, F1 = score(candidate_1, gold, lang="en", verbose=False)
P2, R2, F2 = score(candidate_2, gold, lang="en", verbose=False)

# Round and print
def round_bert(P, R, F):
    return {
        "precision": round(P.item(), 3),
        "recall": round(R.item(), 3),
        "f1": round(F.item(), 3)
    }

bert_1 = round_bert(P1[0], R1[0], F1[0])
bert_2 = round_bert(P2[0], R2[0], F2[0])

print("Candidate 1 (high word overlap):")
print(f"BERTScore: {bert_1}\n")

print("Candidate 2 (semantic similarity, low word overlap):")
print(f"BERTScore: {bert_2}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Candidate 1 (high word overlap):
BERTScore: {'precision': 0.965, 'recall': 0.961, 'f1': 0.963}

Candidate 2 (semantic similarity, low word overlap):
BERTScore: {'precision': 0.909, 'recall': 0.937, 'f1': 0.923}


While its precision, recall, and F1-scores are slightly lower, BERTScore understands that despite different wording, the meaning of candidate 2 closely matches the gold summary.